In [2]:
from pathlib import Path

import dedupe
import polars as pl

In [4]:
DATASET_PATH = Path("datasets/features_dataset_20260706.parquet")

# Chargement du jeu de données

In [5]:
df_features = pl.read_parquet(DATASET_PATH)

In [8]:
df_features.shape

(2000, 84)

In [23]:
df_train = df_features.filter(pl.col("split") == "train")
df_test = df_features.filter(pl.col("split") == "test")
print(df_train.shape, df_test.shape)

(1600, 84) (400, 84)


# Transformation du jeu de données au format dedupe

In [14]:
FEATURES_NAMES = [
    "nom",
    "description",
    "acteur_type_id",
    "adresse",
    "adresse_complement",
    "code_postal",
    "ville",
    "url",
    "email",
    "telephone",
    "nom_commercial",
    "nom_officiel",
    "siren",
    "siret",
    "naf_principal",
    "horaires_osm",
    "horaires_description",
    "horaires_osm",
    "public_acceuilli",
    "reprise",
    "exclusivite_de_reprisereparation",
    "reprise",
    "uniquement_sur_rdv",
    "consignes_dacces",
    "action_principale_id",
    "lieu_prestation",
    "location",
    "code_commune_insee",
    "epci_id",
    "action_reparer",
    "action_acheter",
    "action_revendre",
    "action_donner",
    "action_louer",
    "action_mettreenlocation",
    "action_emprunter",
    "action_preter",
    "action_echanger",
    "action_trier",
    "action_rapporter",
]

In [30]:
def transform_dataset_to_dedupe_format(
    df_features: pl.DataFrame,
) -> tuple[list[tuple[dict]]]:

    match_pairs = []
    distinct_pairs = []

    for row in df_features.iter_rows(named=True):
        actors = []
        for suffix in ["_i", "_j"]:
            actor_formatted = {
                k.replace(suffix, ""): v
                for k, v in row.items()
                if k.replace(suffix, "") in FEATURES_NAMES
            }
            actor_formatted["location"] = (
                row[f"latitude{suffix}"],
                row[f"longitude{suffix}"],
            )
            actors.append(actor_formatted)

        actors = tuple(actors)
        if row["label"]:
            match_pairs.append(actors)
        else:
            distinct_pairs.append(actors)

    return match_pairs, distinct_pairs

In [31]:
match_pairs, distinct_pairs = transform_dataset_to_dedupe_format(df_train)

# Configuration du modèle

In [46]:
variables = [
    dedupe.variables.String(name="nom",field="nom"),
    dedupe.variables.Text("description", has_missing=True),
    dedupe.variables.Categorical(
        "acteur_type_id",
        categories=["1", "2", "3", "4", "5", "7", "8", "9", "10", "11", "12"],
    ),
    dedupe.variables.String("adresse", has_missing=True),
    dedupe.variables.String("adresse_complement", has_missing=True),
    dedupe.variables.Exact("code_postal", has_missing=True),
    dedupe.variables.String("ville", has_missing=True),
    dedupe.variables.Exact("url", has_missing=True),
    dedupe.variables.Exact("email", has_missing=True),
    dedupe.variables.Exact("telephone", has_missing=True),
    dedupe.variables.String("nom_commercial", has_missing=True),
    dedupe.variables.String("nom_officiel", has_missing=True),
    dedupe.variables.Exact("siren", has_missing=True),
    dedupe.variables.Exact("siret", has_missing=True),
    dedupe.variables.Exact("naf_principal", has_missing=True),
    dedupe.variables.Text("horaires_osm", has_missing=True),
    dedupe.variables.Text("horaires_description", has_missing=True),
    dedupe.variables.Categorical(
        "public_acceuilli",
        categories=[
            "Aucun",
            "Particuliers",
            "Particuliers et professionnels",
            "Professionnels",
        ],
        has_missing=True,
    ),
    dedupe.variables.Categorical(
        "reprise", categories=["1 pour 0", "1 pour 1"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "exclusivite_de_reprisereparation", categories=[True, False], has_missing=True
    ),
    dedupe.variables.Categorical(
        "uniquement_sur_rdv", categories=[True, False], has_missing=True
    ),
    dedupe.variables.Text("consignes_dacces", has_missing=True),
    dedupe.variables.Categorical(
        "action_principale_id",
        categories=["7", "8", "5", "6", "1", "9", "3", "2", "12", "11", "4"],
        has_missing=True,
    ),
    dedupe.variables.Categorical(
        "lieu_prestation",
        categories=["A_DOMICILE", "SUR_PLACE", "SUR_PLACE_OU_A_DOMICILE"],
        has_missing=True,
    ),
    dedupe.variables.LatLong("location"),
    dedupe.variables.Exact("code_commune_insee", has_missing=True),
    dedupe.variables.Exact("epci_id", has_missing=True),
    dedupe.variables.Categorical(
        "action_reparer", categories=[True, False], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_acheter", categories=[True, False], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_revendre", categories=[True, False], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_donner", categories=[True, False], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_louer", categories=[True, False], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_mettreenlocation", categories=[True, False], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_emprunter", categories=[True, False], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_preter", categories=[True, False], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_echanger", categories=[True, False], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_trier", categories=[True, False], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_rapporter", categories=[True, False], has_missing=True
    ),
    dedupe.variables.Interaction(
        "nom",
        "nom_commercial",
        "nom_officiel",
    ),
    dedupe.variables.Interaction(
        "adresse",
        "adresse_complement",
        "code_postal",
        "ville",
    ),
    dedupe.variables.Interaction("horaires_osm", "horaires_description"),
    dedupe.variables.Interaction("reprise", "exclusivite_de_reprisereparation"),
]

# Entrainement

In [47]:
model = dedupe.Dedupe(variable_definition=variables)

KeyError: 'The interaction variable nom_commercial is not a named variable in the variable definition'